# 第3回A 演習: 言語モデルの仕組みと実装

スライド: [第3回A](../03a-llm/)

Colab で `ランタイム → ランタイムのタイプを変更 → T4 GPU` にしておく。
CPU でも動くが、生成に時間がかかる。

モデルの取得に数分かかる。先に先頭のセルを実行し、待っている間に課題を読む。

モデルを呼ぶ部分 (`next_scores`) は用意してある。
自分で書くのは普通の Python (リスト・`math`・`random`) だけで、PyTorch は使わない。

## 進め方

- `TODO` を埋めて、直下の確認セルを実行する
- 確認セル (`# [CHECK]`) は自分用。消しても教員の採点は別途走るが、残すこと

| 課題 | 配点 |
|---|---|
| 課題1 スコアを確率にする (softmax) | 10 |
| 課題1-b 次のトークンの上位候補 | 10 |
| 課題2 greedy 生成 | 20 |
| 課題2-b 繰り返し率 | 5 |
| 課題3 温度と top-k / top-p | 20 |
| 課題4 chat | 10 |
| 課題4-b 会話を続ける | 5 |
| 課題5 簡易チャットアプリ | 出力で確認 |
| 発展1〜4 (繰り返し検出 / 圧縮 / ストリーミング / エフォート) | 各5 |

---

## セットアップ

下の2つのセルを上から実行する。モデルの取得に数分かかる。

1. **ライブラリ更新**: Qwen3.5 には最新の transformers が必要 (古い版のままでは動かない)
2. **モデル読み込み**: Qwen3.5-0.8B を読み込む (GPU があると速い)


In [ ]:
# Qwen3.5 は最新の transformers が必要
!pip install -q -U transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL = 'Qwen/Qwen3.5-0.8B'  # マルチモーダル。テキストのみで使う

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(MODEL)
model.eval()

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
model.to(device)
print(device, '/ 語彙数:', len(tok))

# ---- 以降で使うヘルパ (中身は PyTorch。読まなくてよい) ----
def encode(text):
    """文字列 -> トークン ID のリスト"""
    return tok(text).input_ids


def next_scores(ids, n=100):
    """トークン ID のリストから、次に来るトークンの候補を返す。

    [(トークンID, スコア), ...] を、スコアの高い順に n 件。
    スコアは確率ではない。softmax を通すと確率になる (課題1)。
    """
    with torch.no_grad():
        logits = model(torch.tensor([ids], device=device)).logits[0, -1].float()
    top = torch.topk(logits, n)
    return list(zip(top.indices.tolist(), top.values.tolist()))


print(encode('こんにちは'))
print(next_scores(encode('こんにちは'))[:3])

In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認の準備 (check / score)。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '03a'


---
## 課題1: スコアを確率にする (10点)

モデルが返すのは確率ではなく「スコア」のリスト。
これを確率 (合計が 1 のリスト) にする `softmax(scores, temperature=1.0)` を書け。

1. 各スコアを温度で割る: `s / temperature`
2. `math.exp` を取る
3. 合計で割る

**ヒント**: スコアが大きいと `math.exp` があふれる。
先に**最大値を引いて**から `exp` を取ること。引いても確率は変わらない。

### 例題

次のセルを実行して、動きを見る。

`next_scores` が返す「(トークンID, スコア) のリスト」と、`exp` で割合にする流れ。

In [ ]:
# 例題
import math

scores = next_scores(encode('日本の首都は'))
print(scores[:3])    # [(100562, 22.4), (3583, 20.9), ...]

xs = [1.0, 2.0, 3.0]
es = [math.exp(x) for x in xs]
print([e / sum(es) for e in es])    # [0.09, 0.24, 0.67]

温度で割ってから `exp` を取ると、温度が低いほど差が開く。
`softmax([3.0, 1.0], 0.1)` はほぼ `[1.0, 0.0]`、`softmax([3.0, 1.0], 2.0)` は `[0.73, 0.27]`。

In [ ]:
import math


def softmax(scores, temperature=1.0):
    xs = [s / temperature for s in scores]
    m = max(xs)
    es = [math.exp(x - m) for x in xs]
    return [e / sum(es) for e in es]


print(softmax([1.0, 2.0, 3.0]))
print(softmax([3.0, 1.0], 0.1))
print(softmax([3.0, 1.0], 2.0))

In [ ]:
# [CHECK] q1
need('softmax', arity=2)

p = softmax([1.0, 1.0])
assert len(p) == 2 and all(abs(x - 0.5) < 1e-9 for x in p), p

p = softmax([1.0, 2.0, 3.0])
assert abs(sum(p) - 1.0) < 1e-9, '合計が 1 になっていない'
assert p == sorted(p), '大きいスコアほど大きい確率にすること'
assert abs(p[2] - 0.6652409557748219) < 1e-9, p

# 温度: 低いと決め打ち、高いとばらける
assert softmax([3.0, 1.0], 0.1)[0] > 0.99
assert abs(softmax([3.0, 1.0], 2.0)[0] - 0.7310585786300049) < 1e-9

# 大きいスコアでもあふれない
p = softmax([1000.0, 1000.0])
assert abs(p[0] - 0.5) < 1e-9, 'あふれた (最大値を引いてから exp を取る)'
print('OK')

---
### 課題1-b: 次のトークンの候補を見る (10点)

文字列を受け取り、次に来るトークンの**上位 n 候補**を
`[(トークン文字列, 確率), ...]` の形で返す `top_candidates(text, n=10)` を書け。

- 確率の**降順**に並べる
- 確率は `float` にすること
- `next_scores` が返す100件の中での割合でよい

**ヒント**: `next_scores(encode(text))` は高い順なので、並べ替えは要らない。
スコアだけ取り出して `softmax` に渡し、ID を `tok.decode([i])` で文字列に戻す。

### 例題

次のセルを実行して、動きを見る。

タプルのリストから、ID とスコアをそれぞれ取り出す。

In [ ]:
# 例題
scores = next_scores(encode('日本の首都は'))
ids = [i for i, _ in scores]
vals = [s for _, s in scores]
print(ids[:3], vals[:3])
print(repr(tok.decode([ids[0]])))    # '東京'

あとは `softmax(vals)` で確率にして、先頭 `n` 件を `(文字列, 確率)` に組み立てる。
`zip(ids, probs)` で ID と確率を対にできる。

In [ ]:
def top_candidates(text, n=10):
    sc = next_scores(encode(text))
    ps = softmax([s for _, s in sc])
    return [(tok.decode([i]), p) for (i, _), p in zip(sc, ps)][:n]


for t, p in top_candidates('むかしむかし、あるところに'):
    print(f'{t!r:20} {p:.4f}')

In [ ]:
# [CHECK] q1b
need('top_candidates', arity=2)

cand = top_candidates('日本の首都は', n=10)
assert isinstance(cand, list) and len(cand) == 10, len(cand)
assert all(isinstance(t, str) for t, _ in cand), 'トークンは文字列で返すこと'
probs = [float(p) for _, p in cand]
assert probs == sorted(probs, reverse=True), '確率の降順になっていない'
assert 0 < probs[0] <= 1.0, probs[0]
assert sum(probs) <= 1.0001, '確率の合計が1を超えている (softmax を通したか)'
assert len(top_candidates('こんにちは', n=3)) == 3

# 上位1件は、最高スコアの候補と一致するはず
top_id = next_scores(encode('日本の首都は'))[0][0]
assert cand[0][0] == tok.decode([top_id]), '上位1件が最高スコアの候補と違う'
print('OK')

---
## 課題2: greedy 生成 (20点)

毎回もっとも確率の高いトークンを選び続ける `generate_greedy(prompt, max_new_tokens)` を書け。

- 終端トークン (`tok.eos_token_id`) が出たら止める
- 返すのは**プロンプトを含めた全体**の文字列 (`skip_special_tokens=True`)

**ヒント**: `next_scores(ids)` は高い順なので、先頭 `[0][0]` が一番高いトークンの ID。
`ids.append(next_id)` で末尾に足す。

### 例題

次のセルを実行して、動きを見る。

1歩だけ進めてみる。ループはこれの繰り返し。

In [ ]:
# 例題
ids = encode('日本で一番高い山は')
print(ids)    # [101293, ...]

scores = next_scores(ids)
next_id = scores[0][0]
print(next_id, repr(tok.decode([next_id])))

ids.append(next_id)
print(len(ids))
print(tok.decode(ids, skip_special_tokens=True))

終わりの判定は `if next_id == tok.eos_token_id: break`。
これを `ids` にくっつける**前**に置くこと。

In [ ]:
def generate_greedy(prompt, max_new_tokens=40):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        next_id = next_scores(ids)[0][0]
        if next_id == tok.eos_token_id:
            break
        ids.append(next_id)
    return tok.decode(ids, skip_special_tokens=True)


print(generate_greedy('むかしむかし、あるところに'))

In [ ]:
# [CHECK] q2
need('generate_greedy', arity=2)

a = generate_greedy('日本で一番高い山は', 20)
b = generate_greedy('日本で一番高い山は', 20)
assert a == b, 'greedy なのに結果が変わっている'
assert a.startswith('日本で一番高い山は'), a[:30]
assert len(a) > len('日本で一番高い山は'), '何も生成されていない'

# max_new_tokens を守ること
short = generate_greedy('むかしむかし', 3)
n_new = len(tok(short).input_ids) - len(tok('むかしむかし').input_ids)
assert n_new <= 3, f'{n_new} トークン生成された'
assert generate_greedy('あ', 0) == 'あ'
print('OK')


### 課題2-b: 繰り返し率を測る (5点)

greedy は同じ言い回しのループに落ちやすい。それを**数値で**測る関数を書け。

`repetition_rate(text, n=3)` — テキストを空白で区切った単語列について、
**長さ `n` の連続した並び (n-gram) のうち、2回以上現れるものの割合**を返す。

```
repetition_rate('a b c a b c a b c', n=3)
  3-gram: (a,b,c) (b,c,a) (c,a,b) (a,b,c) (b,c,a) (c,a,b) (a,b,c)   全7個
  2回以上現れる位置: 全部 → 1.0
```

- n-gram が1つも作れないときは `0.0` を返す
- 返り値は `0.0` 以上 `1.0` 以下の `float`

**ヒント**: `Counter` で n-gram を数え、`count >= 2` のものが全体のいくつを占めるか。

### 例題

次のセルを実行して、動きを見る。

n-gram の作り方と数え方。


In [ ]:
# 例題
from collections import Counter

ws = 'a b c a b'.split()
grams = [tuple(ws[i:i + 2]) for i in range(len(ws) - 2 + 1)]
print(grams)          # [('a','b'), ('b','c'), ('c','a'), ('a','b')]
print(Counter(grams)) # Counter({('a','b'): 2, ('b','c'): 1, ('c','a'): 1})


あとは「`count >= 2` の gram が、全 4 個のうちいくつを占めるか」。
この例なら 2/4 = 0.5。
`len(ws) - n + 1` が 0 以下のときは gram が作れないので `0.0` を返す。


In [ ]:
from collections import Counter


def repetition_rate(text, n=3):
    ws = text.split()
    g = [tuple(ws[i:i+n]) for i in range(len(ws)-n+1)]
    if not g: return 0.0
    c = Counter(g)
    return sum(1 for x in g if c[x] >= 2) / len(g)


# 実際に測ってみる
out = generate_greedy('好きな食べ物は', 80)
print(out)
print('繰り返し率:', repetition_rate(out))


In [ ]:
# [CHECK] q2b
need('repetition_rate', arity=2)

assert repetition_rate('a b c a b c a b c', n=3) == 1.0
assert repetition_rate('a b c d e f g', n=3) == 0.0
assert repetition_rate('', n=3) == 0.0
assert repetition_rate('a b', n=3) == 0.0, 'n-gram が作れないときは 0.0'
assert repetition_rate('a b c', n=3) == 0.0, '1個だけなら繰り返しではない'

r = repetition_rate('x y z x y z p q r', n=3)
assert 0.0 < r < 1.0, r
assert abs(r - 2/7) < 1e-9, f'{r} (7個の3-gram のうち (x,y,z) だけが2回)'

assert repetition_rate('a a a a', n=1) == 1.0
assert isinstance(repetition_rate('a b c a b c', n=2), float)

# 性質: 割合なので必ず 0 以上 1 以下
holds('0 以上 1 以下',
      lambda ws: 0.0 <= repetition_rate(' '.join(ws), 2) <= 1.0,
      [words(n) for n in range(0, 12)])
holds('同じ語だけ並べたら 1.0',
      lambda n: repetition_rate(' '.join(['x'] * n), 2) == 1.0, range(4, 12))
print('OK')


---
## 課題3: 温度と top-k / top-p (20点)

温度付き top-k の `sample_next(scores, temperature, k)` と、
累積確率 top-p の `sample_next_top_p(scores, temperature, p)` を書け。
どちらも返すのは**トークン ID (int)** 。`scores` は `next_scores` の返り値。

1. 上位 `k` 個だけを候補にする (`scores` は高い順なので `scores[:k]`)
2. 課題1 の `softmax(..., temperature)` で確率にする
3. その確率に比例して1つ引く

**ヒント**: `random.choices(候補のリスト, weights=確率のリスト)[0]` が重み付きのくじ引き。
第2回の `pick_next` と同じ形。

top-p は、確率の高い順に足していき、**足す前の合計が `p` 未満の間**だけ候補に残す。

使うもの: `softmax` / スライス / `random.choices`

In [ ]:
# 例題
import random

scores = next_scores(encode('日本の首都は'))
top5 = scores[:5]
ids = [i for i, _ in top5]
probs = softmax([s for _, s in top5], 0.8)
print(ids)                                    # [100562, 3583, ...]
print(probs)                                  # [0.61, 0.20, ...]
print(random.choices(ids, weights=probs)[0])

温度の効き方はこうなる。

In [ ]:
# 例題
print(softmax([3.0, 1.0], 0.1))    # [1.00, 0.00]
print(softmax([3.0, 1.0], 2.0))    # [0.73, 0.27]

In [ ]:
import random


def sample_next(scores, temperature=0.8, k=50):
    top = scores[:k]
    probs = softmax([s for _, s in top], temperature)
    return random.choices([i for i, _ in top], weights=probs)[0]


def generate(prompt, max_new_tokens=60, temperature=0.8, k=50):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        next_id = sample_next(next_scores(ids), temperature, k)
        if next_id == tok.eos_token_id:
            break
        ids.append(next_id)
    return tok.decode(ids, skip_special_tokens=True)


for t in [0.3, 0.8, 1.5]:
    print(f'--- temperature = {t} ---')
    print(generate('プログラミングを学ぶと', max_new_tokens=40, temperature=t))
    print()

In [ ]:
# [CHECK] q3
need('sample_next', arity=3)

scores = next_scores(encode('日本の首都は'))

# k=1 なら greedy と一致するはず
for _ in range(5):
    assert sample_next(scores, temperature=1.0, k=1) == scores[0][0]
assert isinstance(sample_next(scores, 1.0, 50), int), 'トークン ID (int) を返すこと'

# 候補は必ず上位 k 個の中から出る
top50 = {i for i, _ in scores[:50]}
for _ in range(30):
    assert sample_next(scores, 1.0, 50) in top50
top5 = {i for i, _ in scores[:5]}
for _ in range(30):
    assert sample_next(scores, 1.0, 5) in top5, 'k が効いていない'
for _ in range(30):
    assert sample_next([(7, 0.0), (8, 0.0), (9, 0.0)], 1.0, 2) in (7, 8), 'k が効いていない'

# 温度を上げれば、選ばれるトークンの種類が増えるはず
cold = {sample_next(scores, 0.1, 50) for _ in range(30)}
hot  = {sample_next(scores, 2.0, 50) for _ in range(30)}
assert len(hot) > len(cold), f'低温 {len(cold)} 種 / 高温 {len(hot)} 種'

# 生成が動くこと
g = generate('むかしむかし', max_new_tokens=10, temperature=0.8)
assert isinstance(g, str) and g.startswith('むかしむかし')
print('OK')

---

### top-p の例題

確率の高い順に足していき、**足す前の合計**が `p` 未満の間だけ残す。

```
出力例 (p = 0.9):
0 足す前の合計 0.00 → 残す
1 足す前の合計 0.50 → 残す
2 足す前の合計 0.80 → 残す
3 足す前の合計 0.95 → 捨てる
```

In [ ]:
# 例題
probs = [0.5, 0.3, 0.15, 0.05]
cum = 0.0
for i, pr in enumerate(probs):
    print(i, f'足す前の合計 {cum:.2f}', '→', '残す' if cum < 0.9 else '捨てる')
    cum += pr

自分を足した後の合計ではなく、**足す前の合計**と比べるのがコツ。
こうすると、最上位は必ず残る (足す前は 0) ので候補が空にならない。
あとは残った候補だけで、top-k と同じように `softmax` → `random.choices` で1つ引く。

In [ ]:
def sample_next_top_p(scores, temperature=0.8, p=0.9):
    probs = softmax([s for _, s in scores], temperature)
    keep, cum = 0, 0.0
    for pr in probs:
        if cum >= p:
            break
        keep += 1
        cum += pr
    return random.choices([i for i, _ in scores[:keep]], weights=probs[:keep])[0]

In [ ]:
# [CHECK] q3p
need('sample_next_top_p', arity=3)

scores = next_scores(encode('日本の首都は'))

# p が極小なら、実質 greedy
for _ in range(5):
    assert sample_next_top_p(scores, 1.0, 1e-6) == scores[0][0]

# p=0.9 の候補は、累積 0.9 に届くまでの語だけ
import math as _m
_mx = scores[0][1]
_es = [_m.exp(s - _mx) for _, s in scores]
allowed, cum = set(), 0.0
for (i, _), e in zip(scores, _es):
    if cum < 0.9:
        allowed.add(i)
    cum += e / sum(_es)
for _ in range(30):
    assert sample_next_top_p(scores, 1.0, 0.9) in allowed

# 確率が等しい4つ、p=0.5 なら上位2つだけ
for _ in range(30):
    assert sample_next_top_p([(1, 0.0), (2, 0.0), (3, 0.0), (4, 0.0)], 1.0, 0.5) in (1, 2)
print('OK')

---
## 課題4: chat (10点)

チャットテンプレートを使って、1回の応答を返す `chat(messages)` を書け。

1. `tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)` で
   プロンプト文字列を作る
2. `sample_next` を使って生成する
3. **新しく生成した部分だけ**を文字列で返す

**ヒント**: 生成前の `len(ids)` を覚えておき、最後に `ids[start:]` を decode する。
停止判定は `next_id in tok.all_special_ids`。

### 例題

次のセルを実行して、動きを見る。

まず、テンプレートが何を作るのかを見る。

In [ ]:
# 例題
p = tok.apply_chat_template(
    [{'role': 'user', 'content': 'やあ'}],
    tokenize=False, add_generation_prompt=True)
print(repr(p))
# '<|im_start|>system\n...<|im_end|>\n<|im_start|>user\nやあ<|im_end|>\n
#  <|im_start|>assistant\n'


つまり `chat` は「この文字列の続きを生成する」だけ。
生成のループは課題3 の `generate` とまったく同じ。違いは2つ。


In [ ]:
# 例題
start = len(ids)
# ... ここで課題3 と同じループ ...
tok.decode(ids[start:], skip_special_tokens=True)

止まる判定は `if next_id in tok.all_special_ids: break`。
`<|im_end|>` も特殊トークンなので、これで応答の終わりを捕まえられる。

In [ ]:
def chat(messages, max_new_tokens=60, temperature=0.7, k=50):
    prompt = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    ids = encode(prompt)
    start = len(ids)
    for _ in range(max_new_tokens):
        nid = sample_next(next_scores(ids), temperature, k)
        if nid in tok.all_special_ids:
            break
        ids.append(nid)
    return tok.decode(ids[start:], skip_special_tokens=True)


msgs = [
    {'role': 'system', 'content': 'あなたは簡潔に答えるアシスタントです。'},
    {'role': 'user', 'content': '日本の首都はどこですか？'},
]
print(chat(msgs))

In [ ]:
# [CHECK] q4
need('chat', arity=1)

reply = chat([{'role': 'user', 'content': '1 + 1 は？'}], max_new_tokens=30)
assert isinstance(reply, str) and reply.strip(), '空の応答'
assert '1 + 1 は？' not in reply, 'プロンプトが混ざっている (start の位置を見直す)'
assert '<|im_start|>' not in reply and '<|im_end|>' not in reply, '特殊トークンが残っている'
assert 'system' not in reply.lower()[:20], 'テンプレート全体を返している'

# max_new_tokens を守ること
short = chat([{'role': 'user', 'content': '長い昔話をして'}], max_new_tokens=5)
assert len(tok(short).input_ids) <= 6, f'{len(tok(short).input_ids)} トークン返った'
print('OK')


### 課題4-b: 会話を続ける (5点)

`chat_session(user_lines)` を書け。
ユーザの発言のリストを順に投げ、**履歴を積み重ねながら**応答を集めて返す。

- 返り値は応答文字列のリスト (長さは `user_lines` と同じ)
- 2回目以降は、それまでの `user` と `assistant` の発言をすべて含めて `chat` に渡す
- system メッセージは先頭に1つだけ

**採点できるように `input()` は使わないこと。** 対話したいときは下のセルを使う。

**ヒント**: 履歴のリストに `{'role': 'user', ...}` と
`{'role': 'assistant', ...}` を交互に足していく。

### 例題

次のセルを実行して、動きを見る。

履歴に交互に足していくだけ。


In [ ]:
# 例題
history = [{'role': 'system', 'content': SYSTEM}]

history.append({'role': 'user', 'content': 'こんにちは'})
r = chat(history)
history.append({'role': 'assistant', 'content': r})

history.append({'role': 'user', 'content': '元気？'})
r2 = chat(history)

課題は、これを `for u in user_lines:` のループにしたもの。
`replies` には `r` だけを足していく。


In [ ]:
SYSTEM = '簡潔に答えてください。'


def chat_session(user_lines, max_new_tokens=40):
    history = [{'role': 'system', 'content': SYSTEM}]
    replies = []
    for u in user_lines:
        history.append({"role": "user", "content": u})
        r = chat(history, max_new_tokens=max_new_tokens)
        history.append({"role": "assistant", "content": r})
        replies.append(r)
    return replies


for u, r in zip(['犬と猫はどちらが好き？', 'なぜ？', '他には？'],
                chat_session(['犬と猫はどちらが好き？', 'なぜ？', '他には？'])):
    print('あなた:', u)
    print('AI    :', r)
    print()


In [ ]:
# 対話用 (採点しない)。実行すると入力待ち
def chat_interactive():
    history = [{'role': 'system', 'content': SYSTEM}]
    while True:
        user = input('あなた: ')
        if user.strip() in ('quit', 'exit', ''):
            break
        history.append({'role': 'user', 'content': user})
        reply = chat(history)
        print('AI:', reply)
        history.append({'role': 'assistant', 'content': reply})


# chat_interactive()   ← コメントを外すと対話できる


In [ ]:
# [CHECK] q4b
need('chat_session', arity=1)

rs = chat_session(['こんにちは', '元気？', 'さようなら'], max_new_tokens=20)
assert isinstance(rs, list) and len(rs) == 3, f'{len(rs)} 件返った'
assert all(isinstance(r, str) and r.strip() for r in rs), '空の応答がある'
assert all('<|im_' not in r for r in rs), '特殊トークンが残っている'
assert chat_session([]) == []

# 履歴が積み上がっていることを、渡された messages の長さで確かめる
seen = []
_orig = chat
def _spy(messages, **kw):
    seen.append([m['role'] for m in messages])
    return _orig(messages, **kw)
chat = _spy
try:
    chat_session(['A', 'B', 'C'], max_new_tokens=8)
finally:
    chat = _orig
assert len(seen) == 3, f'chat が {len(seen)} 回しか呼ばれていない'
assert seen[0].count('user') == 1, seen[0]
assert seen[1].count('user') == 2 and seen[1].count('assistant') == 1, seen[1]
assert seen[2].count('user') == 3 and seen[2].count('assistant') == 2, seen[2]
assert all(r[0] == 'system' and r.count('system') == 1 for r in seen), 'system は先頭に1つ'
print('OK')


---

## 課題5: 簡易チャットアプリを作る

課題4 の `chat(messages, ...)` を使って、履歴を保持する「チャットボット」
`make_chat()` を作ろう。履歴は `make_chat()` の中だけに閉じ込め、毎回全部渡す。

- `make_chat(system_prompt)` は `say(text)` を返す
- `say` は `chat(history, ...)` で応答を得て、履歴に足す
- 3往復以上、好きな話題で会話する

使うもの: `chat(messages, **opts)` と、関数の中に持たせた `history`

採点は確認セルでなく、出力 (会話の記録) で行う。

In [ ]:
# 課題5: 簡易チャットアプリを作ろう
def make_chat(system_prompt='簡潔に答えてください。'):
    history = [{'role': 'system', 'content': system_prompt}]

    def say(text, **opts):
        history.append({'role': 'user', 'content': text})
        # TODO: chat(history, max_new_tokens=48) で応答を得る
        reply = ...
        # TODO: history に assistant の応答を足して、応答を返す
        ...
        return reply

    return say


bot = make_chat()
for q in ['日本の首都は？', 'そこの人口は？', '有名な食べ物は？']:
    print('Q:', q)
    print('A:', bot(q))
    print()

---

# 発展課題

余力があれば。課題5 の `make_chat` を、発展1〜4 で段階的に改良していく。
それぞれまず「部品となる純関数」を書き、それを `make_chat` に組み込む。
部品は `score()` で採点される。

---

## 発展1: 繰り返しの検出 (5点)

応答が同じフレーズを繰り返すと単調になる。テキストの中に、
n 語のまとまりが2回以上現れたら `True` を返す `has_repeat(text, n)` を書け。

**ヒント**: 先頭から n 語ずつのチャンクを作り、「見たことがあるか」を集合に覚えておく。

**チャンクと set の早見表**:

```python
words = text.split()          # 文字列 -> 語のリスト
chunk = tuple(words[i:i + n]) # n 語のまとまり (スライス)。タプルにすると set に入れられる
seen = set()                  # 「見たことがあるもの」の集合
chunk in seen                 # 含まれているか (True / False)
seen.add(chunk)               # 追加
```

使うもの: `text.split()` と 集合 (`set`) または `in` 判定

In [ ]:
def has_repeat(text, n=3):
    w = text.split(); seen = set()
    for i in range(len(w)-n+1):
        c = tuple(w[i:i+n])
        if c in seen: return True
        seen.add(c)
    return False


print(has_repeat('I like cats. I like cats. Yes', 3))  # True
print(has_repeat('the quick brown fox', 3))             # False

In [ ]:
# [CHECK] e1
need('has_repeat', arity=2)

assert has_repeat('I like cats. I like cats. Yes', 3)
assert not has_repeat('the quick brown fox', 3)
assert has_repeat('la la la', 1)
assert not has_repeat('', 3)
print('OK')

---

## 発展2: コンテキストと圧縮 (5点)

会話が長くなると入力上限に近づく。履歴のトークン数を概算する
`estimate_tokens(messages)` と、上限を超えたら古いやりとりを捨てる
`compact(messages, limit)` を書け。`system` は残すこと。

**ヒント**: 1トークン ≈ 日本語2文字とみなして概算してよい。
スライス `messages[1:3]` で1往復を捨てる。

使うもの: `len(...)` による文字数概算、スライス

In [ ]:
def estimate_tokens(messages):
    return sum(len(m["content"]) for m in messages) // 2 + 1


def compact(messages, limit):
    m = list(messages)
    while estimate_tokens(m) > limit and len(m) > 1:
        m = m[:1] + m[2:]
    return m


msgs = [
    {'role': 'system', 'content': 'あなたは簡潔に答えるアシスタントです。'},
    {'role': 'user', 'content': 'あ' * 200},
    {'role': 'assistant', 'content': 'い' * 200},
    {'role': 'user', 'content': 'う'},
]
print(len(compact(msgs, 50)))    # 1

In [ ]:
# [CHECK] e2
need('estimate_tokens', arity=1)
need('compact', arity=2)

assert estimate_tokens([{'role': 'user', 'content': 'あいうえお'}]) >= 1
msgs = [
    {'role': 'system', 'content': 's'},
    {'role': 'user', 'content': 'あ' * 200},
    {'role': 'assistant', 'content': 'い' * 200},
    {'role': 'user', 'content': 'う'},
]
c = compact(msgs, 50)
assert c[0]['role'] == 'system'
assert len(c) < len(msgs)
print('OK')

---

## 発展3: ストリーミング表示 (5点)

生成された断片のリストを1つの文字列にまとめる `collect(pieces)` を書け。

使うもの: 文字列の連結 (`''.join(pieces)` か `+=`)

In [ ]:
def collect(pieces):
    return ''.join(pieces)


print(collect(['む', 'か', 'し']))  # むかし

In [ ]:
# [CHECK] e3
need('collect', arity=1)

assert collect(['む', 'か', 'し']) == 'むかし'
assert collect([]) == ''
assert collect(['a', 'b', 'c']) == 'abc'
print('OK')

---

## 発展4: エフォート (low / medium / high) (5点)

生成の「丁寧さ」を effort で切り替える。effort ごとの設定を返す
`effort_opts(effort)` を書け。

**ヒント**: low は短く・ばらける (温度高め)、high は長く・堅く (温度低め)。

使うもの: dict の参照 `EFFORT[effort]`

In [ ]:
EFFORT = {
    'low':    {'max_new_tokens': 32,  'temperature': 1.0},
    'medium': {'max_new_tokens': 64,  'temperature': 0.8},
    'high':   {'max_new_tokens': 128, 'temperature': 0.3},
}

def effort_opts(effort):
    return EFFORT[effort]

In [ ]:
# [CHECK] e4
need('EFFORT', kind=dict)

assert set(EFFORT) == {'low', 'medium', 'high'}
need('effort_opts', arity=1)
o = effort_opts('high')
assert 'max_new_tokens' in o and 'temperature' in o
assert effort_opts('high')['max_new_tokens'] > effort_opts('low')['max_new_tokens']
assert effort_opts('low')['temperature'] > effort_opts('high')['temperature']
print('OK')

In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

提出は kibaco から。`score()` と採点ページは下書き確認用。

1. 上から実行し、出力を残す
2. `ファイル → .ipynb をダウンロード`
3. 締切までに kibaco へ出す

確認セルが全部 `OK` なら提出してよい。確認セルを消しても教員の採点は別の検査で行う。
